In [ ]:
# Cell 1: resolve project root and import dependencies
import json
import sys
from pathlib import Path

import d4rl  # noqa: F401
import gym
import numpy as np
import torch
from torch.utils.data import DataLoader


def find_project_root(start: Path) -> Path:
    """Walk upward until the repository root containing src/ is found."""
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate the project root containing 'src/'.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ARTIFACTS_DIR, CHECKPOINTS_DIR, OBS_STATS_DIR, RAW_METRICS_DIR
from src.dataset import NoisyOfflineRLDataset
from src.encoder import PlainEncoder
from src.experiment_config import *
from src.iql import IQLAgent
from src.train_eval import (
    eval_policy_on_env,
    load_and_freeze_encoder,
    save_metrics_json,
    train_iql_from_loader,
)


In [ ]:
# Cell 2: experiment configuration and output directories
METHOD = "plain"


def scale_tag(value):
    """Convert a float into a filename-safe tag, e.g. 0.5 -> 0p5."""
    return str(value).replace(".", "p")


def setting_tag(noise_dim, noise_scale, noise_type):
    """Build the experiment setting identifier used in output paths."""
    return f"nd{noise_dim}_ns{scale_tag(noise_scale)}_{noise_type}"


SETTING_TAG = setting_tag(NOISE_DIM, NOISE_SCALE, NOISE_TYPE)
SEED_TAG = f"seed_{SEED}"

CHECKPOINT_DIR = CHECKPOINTS_DIR / METHOD / ENV_NAME / SETTING_TAG / SEED_TAG
METRICS_DIR = RAW_METRICS_DIR / METHOD / ENV_NAME / SETTING_TAG / SEED_TAG
OBS_STATS_RUN_DIR = OBS_STATS_DIR / METHOD / ENV_NAME / SETTING_TAG / SEED_TAG

for directory in [CHECKPOINT_DIR, METRICS_DIR, OBS_STATS_RUN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("DEVICE:", DEVICE)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("METRICS_DIR:", METRICS_DIR)
print("OBS_STATS_RUN_DIR:", OBS_STATS_RUN_DIR)


In [ ]:
# Cell 3: build the dataset and data loaders
dataset = NoisyOfflineRLDataset(
    env_name=ENV_NAME,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    seed=SEED,
    use_timeouts=True,
    noise_type=NOISE_TYPE,
)

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)

state_dim = dataset.noisy_obs.shape[1]
action_dim = dataset.actions.shape[1]
true_state_dim = dataset.obs_dim
LATENT_DIM = int(max(true_state_dim, NOISE_DIM) * 1.5)

print("state_dim (noisy):", state_dim)
print("true_state_dim:", true_state_dim)
print("action_dim:", action_dim)
print("latent_dim (auto):", LATENT_DIM)

# Save observation normalization statistics for reproducible evaluation.
stats_path = OBS_STATS_RUN_DIR / "obs_stats.npz"
if not stats_path.exists():
    np.savez(
        stats_path,
        obs_mean=dataset.obs_mean,
        obs_std=dataset.obs_std,
        true_state_dim=true_state_dim,
    )
    print("Saved:", stats_path)
else:
    print("Already exists:", stats_path)


In [ ]:
# Cell 4: pretrain the plain encoder baseline
# The plain encoder uses:
# - latent prediction for clean next-state dynamics
# - reward prediction from the latent representation
# - no disentanglement or independence regularization

torch.manual_seed(SEED)
np.random.seed(SEED)

encoder = PlainEncoder(
    state_dim=state_dim,
    action_dim=action_dim,
    true_state_dim=true_state_dim,
    latent_dim=LATENT_DIM,
).to(DEVICE)

optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-4)
pretrain_loader = DataLoader(dataset, batch_size=PRETRAIN_BS, shuffle=True, drop_last=True,
                             num_workers=4, pin_memory=True, persistent_workers=True)

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    encoder.train()
    losses = []

    for obs, act, next_obs, rew, done, pure_obs, pure_next_obs in pretrain_loader:
        obs = obs.to(DEVICE)
        act = act.to(DEVICE)
        rew = rew.to(DEVICE)
        pure_next_obs = pure_next_obs.to(DEVICE)

        # PlainEncoder returns (z, None) to match the disentangled interface.
        z_task, _ = encoder(obs)

        pred_next_true = encoder.state_predictor(torch.cat([z_task, act], dim=-1))
        loss_dyn = torch.nn.functional.mse_loss(pred_next_true, pure_next_obs)

        pred_rew = encoder.reward_predictor(z_task)
        loss_rew = torch.nn.functional.mse_loss(pred_rew.squeeze(-1), rew)

        loss = loss_dyn + loss_rew

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optimizer.step()

        losses.append(float(loss.item()))

    print(f"[pretrain] epoch {epoch}, loss={np.mean(losses):.4f}")

CKPT_ENCODER = CHECKPOINT_DIR / f"encoder_epoch_{PRETRAIN_EPOCHS}.pth"
torch.save(encoder.state_dict(), CKPT_ENCODER)
print("Saved encoder:", CKPT_ENCODER)


In [ ]:
# Cell 5: load and freeze the encoder, then train IQL

encoder = load_and_freeze_encoder(
    encoder=encoder,
    ckpt_path=CKPT_ENCODER,
    device=DEVICE,
)

iql = IQLAgent(
    latent_dim=LATENT_DIM,
    action_dim=action_dim,
    device=DEVICE,
    expectile=0.7,
    temperature=3.0,
    discount=0.99,
)

iql_history = train_iql_from_loader(
    encoder=encoder,
    iql=iql,
    train_loader=train_loader,
    device=DEVICE,
    epochs=EPOCHS,
    ckpt_dir=CHECKPOINT_DIR,
    method=METHOD,
    save_every=10,
    use_tqdm=False,
)


In [ ]:
# Cell 6: evaluate the policy and save metrics
print("Start evaluating ...")

metrics = eval_policy_on_env(
    iql=iql,
    env_name=ENV_NAME,
    encoder=encoder,
    method=METHOD,
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    episodes=20,
    max_steps=1000,
    seed=SEED,
    device=DEVICE,
    use_fixed_noise=True,
)

metrics_path = save_metrics_json(
    metrics_dir=METRICS_DIR,
    metrics=metrics,
    env_name=ENV_NAME,
    method=METHOD,
    seed=SEED,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    extra={
        "latent_dim": LATENT_DIM,
        "pretrain_epochs": PRETRAIN_EPOCHS,
        "iql_epochs": EPOCHS,
    },
)

print(metrics)
print("Saved metrics:", metrics_path)
